In [33]:
import numpy
import sys
import nrrd
import tomllib
import meshio
from scipy.interpolate import splev
import trimesh
from scipy.integrate import trapezoid

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import export_centerline


with open("../src/config.toml", mode="rb") as f:
    config = tomllib.load(f)

sequence = "T1"

if "T2":
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file_T2"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename_T2"]
    segment_folder  = config["data"]["segment_folder"]
    out_folder  = config["data"]["centerline_folder"]
    mesh_folder = config["data"]["output_folder"]
else:
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename"]
    segment_folder  = config["data"]["segment_folder"]
    out_folder  = config["data"]["centerline_folder"]
    mesh_folder = config["data"]["output_folder"]



## IDs

In [34]:
ON_id = 7
eyeball_id = 1

h = 0.9

## Load segments, pixel spacing etc

In [35]:
in_folder = base + mesh_folder + "re-meshed_h_"+str(h).replace(".","-") +"_cenetered_"+sequence + "/"

mesh_filename = in_folder + "id_"+str(ON_id)+"__h_"+str(h).replace(".","-")+".stl"  # your .inp mesh file
mesh_filename_eyeball = in_folder + "id_"+str(eyeball_id)+"_shell__h_"+str(h).replace(".","-")+".stl"  # your .inp mesh file

mesh_ON = meshio.read(mesh_filename)
mesh_eyeball = meshio.read(mesh_filename_eyeball)




/opt/homebrew/Caskroom/miniconda/base/envs/eye-project/lib/python3.13/site-packages/meshio/stl/_stl.py:40: RuntimeWarning: overflow encountered in scalar multiply
  if 84 + num_triangles * 50 == filesize_bytes:


In [ ]:
tri = trimesh.Trimesh(
    vertices=mesh_ON.points,
    faces=mesh_ON.cells_dict["triangle"],
    process=True,
)
tri_eye = trimesh.Trimesh(
    vertices=mesh_eyeball.points,
    faces=mesh_eyeball.cells_dict["triangle"],
    process=True,
)


[ 0.11329775 -0.14261529 -0.0251149 ]


## Load centerline

In [37]:
def load_centerline(npz_path):
    d = numpy.load(npz_path)
    tcks = [
        (d["tx"], d["cx"], int(d["kx"])),
        (d["ty"], d["cy"], int(d["ky"])),
        (d["tz"], d["cz"], int(d["kz"])),
    ]
    u0 = max(tck[0][0] for tck in tcks)
    u1 = min(tck[0][-1] for tck in tcks)

    def curve(u, der=0):
        u = numpy.atleast_1d(numpy.asarray(u, dtype=float))
        return numpy.column_stack([splev(u, tck, der=der) for tck in tcks])

    return curve, (u0, u1)

## Find ends

In [38]:
def orient_to_eye(curve, u_a, p_a, u_b, p_b, eye_center):
    """Swaps the ends so that end A is the one closer to the eye."""
    d_a = numpy.linalg.norm(curve(u_a)[0] - eye_center)
    d_b = numpy.linalg.norm(curve(u_b)[0] - eye_center)
    if d_b < d_a:
        return u_b, p_b, u_a, p_a
    return u_a, p_a, u_b, p_b

In [39]:
def find_mesh_exits(curve, u_range, tri_mesh, n_samples=4000, margin=0.1, u_start=None):
    """
    Marches from u_start outward in both directions and returns the first
    crossing with the surface on each side.
    margin: extend the sampled parameter range beyond the stored knot range
            (as a fraction of it) in case the surface lies slightly beyond.
    """
    u0, u1 = u_range
    span = u1 - u0
    u = numpy.linspace(u0 - margin * span, u1 + margin * span, n_samples)
    pts = curve(u)

    if u_start is None:
        u_start = 0.5 * (u0 + u1)
    i_mid = int(numpy.searchsorted(u, u_start))

    p_start = curve(u_start)
    if tri_mesh.is_watertight and not tri_mesh.contains(p_start)[0]:
        print("Warning: start point of the march is not inside the mesh.")

    seg = pts[1:] - pts[:-1]
    seg_len = numpy.linalg.norm(seg, axis=1)
    dirs = seg / seg_len[:, None]

    def first_exit(seg_idx, forward):
        # seg_idx is ordered in marching direction
        origins = pts[seg_idx] if forward else pts[seg_idx + 1]
        directions = dirs[seg_idx] if forward else -dirs[seg_idx]
        locs, ray_idx, _ = tri_mesh.ray.intersects_location(
            origins, directions, multiple_hits=False
        )
        if len(ray_idx) == 0:
            return None, None
        dist = numpy.linalg.norm(locs - origins[ray_idx], axis=1)
        ok = dist <= seg_len[seg_idx[ray_idx]]
        if not numpy.any(ok):
            return None, None
        ray_idx, locs, dist = ray_idx[ok], locs[ok], dist[ok]
        j = numpy.argmin(ray_idx)                 # first crossing in marching order
        s = seg_idx[ray_idx[j]]
        frac = dist[j] / seg_len[s]
        du = u[s + 1] - u[s]
        u_hit = u[s] + frac * du if forward else u[s + 1] - frac * du
        return u_hit, locs[j]

    u_b, p_b = first_exit(numpy.arange(i_mid, len(seg)), forward=True)
    u_a, p_a = first_exit(numpy.arange(i_mid - 1, -1, -1), forward=False)

    if u_a is None or u_b is None:
        raise RuntimeError("No intersection found on one side; increase margin.")
    return u_a, p_a, u_b, p_b

## Define features

In [40]:
def centerline_metrics(curve, u_a, u_b, n=5000, kappa_min=1e-4):
    u = numpy.linspace(u_a, u_b, n)
    pts = curve(u)
    d1, d2, d3 = curve(u, 1), curve(u, 2), curve(u, 3)

    speed = numpy.linalg.norm(d1, axis=1)          # ds/du
    s = numpy.concatenate([[0], numpy.cumsum(0.5 * (speed[1:] + speed[:-1]) * numpy.diff(u))])
    L = s[-1]                                       # arc length

    p_a, p_b = pts[0], pts[-1]
    C = numpy.linalg.norm(p_b - p_a)                # chord ("straight") length

    # curvature and torsion of the parametric curve
    cr = numpy.cross(d1, d2)
    cr_n = numpy.linalg.norm(cr, axis=1)
    kappa = cr_n / speed**3
    tau = numpy.einsum("ij,ij->i", cr, d3) / numpy.maximum(cr_n**2, 1e-30)
    tau[kappa < kappa_min] = 0.0                    # torsion undefined on straight parts

    total_curvature = trapezoid(kappa, s)           # [rad]
    total_torsion = trapezoid(numpy.abs(tau), s)    # [rad]

    # deviation from the chord
    e = (p_b - p_a) / C
    rel = pts - p_a
    perp = rel - (rel @ e)[:, None] * e
    dev = numpy.linalg.norm(perp, axis=1)
    i_max = numpy.argmax(dev)

    # angle between end tangents
    t_a, t_b = d1[0] / speed[0], d1[-1] / speed[-1]
    end_angle = numpy.degrees(numpy.arccos(numpy.clip(t_a @ t_b, -1, 1)))

    # planarity: smallest PCA eigenvalue of the centerline points
    ev = numpy.linalg.eigvalsh(numpy.cov((pts - pts.mean(0)).T))


    return {
        "end_point_a": p_a,
        "end_point_b": p_b,
        "arc_length": L,
        "chord_length": C,
        "distance_metric": L / C,
        "max_chord_deviation": dev[i_max],
        "max_deviation_position": s[i_max] / L,     # fraction 0..1 from eye end
        "max_deviation_arc_mm": s[i_max],           # arc length from eye end [mm]
        "max_deviation_point": pts[i_max],
        "mean_curvature": total_curvature / L,
        "max_curvature": kappa.max(),
    }

## Get features

In [41]:
curve, u_range = load_centerline(base + out_folder + "centerline_id_" + str(ON_id) + ".npz")

u_a, p_a, u_b, p_b = find_mesh_exits(curve, u_range, tri, margin=0.1)
u_a, p_a, u_b, p_b = orient_to_eye(curve, u_a, p_a, u_b, p_b, eye_center)

m = centerline_metrics(curve, u_a, u_b)

print(f"L [mm]: {m['arc_length']:.3f}")
print(f"C [mm]: {m['chord_length']:.3f}")
print(f"Max deviation [mm]: {m['max_chord_deviation']:.3f}")
print(f"Max deviation position [mm]: {m['max_deviation_arc_mm']:.3f}")
print(f"Mean curvature [1/mm]: {m['mean_curvature']:.4f}")
print(f"Max curvature [1/mm]: {m['max_curvature']:.4f}")

L [mm]: 27.937
C [mm]: 27.822
Max deviation [mm]: 0.777
Max deviation position [mm]: 9.185
Mean curvature [1/mm]: 0.0187
Max curvature [1/mm]: 0.0336


In [42]:

export_centerline(
    numpy.array([m["end_point_a"], m["end_point_b"]]), 
    1, 
    base + out_folder  + "chord_id_"+str(ON_id)+".vtp", 
    [0,0,0]
)

export_centerline(
    curve(numpy.linspace(u_a, u_b, 500)),
    1,
    base + out_folder + "centerline_measured_id_"+str(ON_id)+".vtp",
    [0, 0, 0],
)